In [ ]:
# %%capture
!pip install -q -U accelerate peft bitsandbytes transformers trl sentencepiece triton torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.0 MB/s eta 0:00:00


# Supervised Fine-Tuning (SFT)

## Data Preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load a tokenizer to use its chat template
template_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

def format_prompt(example):

    # Format answers
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)

    return {"text": prompt}

# Load and format the data using the template TinyLLama is using
path = "/content/drive/MyDrive/dataset.jsonl"
dataset = load_dataset("json", data_files=path, split="train")
dataset = dataset.map(format_prompt)
print(dataset[0]["text"])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

<|im_start|>system
You are a personalized AI assistant representing Abdelrahman ElKhatteeb.<|im_end|>
<|im_start|>user
Who are you and what is your current educational background?<|im_end|>
<|im_start|>assistant
I am Abdelrahman ElKhatteeb. I am a student at Cairo University pursuing a Bachelor's degree in Computer Science and Artificial Intelligence, with an expected graduation in June 2027.<|im_end|>



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Prepare LoRA Configuration
peft_config = LoraConfig(
    lora_alpha=32,  # LoRA Scaling
    lora_dropout=0.1,  # Dropout for LoRA Layers
    r=64,  # Rank
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=  # Layers to target
     ['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

output_dir = "./results"

# Training arguments
training_arguments = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=10,
    logging_steps=1,
    fp16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
)

In [ ]:
from trl import SFTTrainer

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_arguments,
    peft_config=peft_config,
)

# Train model
trainer.train()

# Save QLoRA weights
trainer.model.save_pretrained("Qwen-1.5B-LoRA")

Tokenizing train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,3.799684
2,2.822748
3,2.692165
4,3.091976
5,2.121837
6,2.126625
7,0.653200
8,1.581728
9,1.463360
10,1.086718


In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

output_dir = "./results"

# Training arguments
training_arguments = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=10,
    logging_steps=1,
    fp16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
)


bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,  # Use 4-bit precision model loading
  bnb_4bit_quant_type="nf4",  # Quantization type
  bnb_4bit_compute_dtype=torch.float16,  # Compute dtype
  bnb_4bit_use_double_quant=True,  # Apply nested quantization
)

try:
    del model
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)

model = prepare_model_for_kbit_training(model)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_arguments,
    peft_config=peft_config,
)

for param in trainer.model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)
# --------------------

# Start training
trainer.train()

# Save the QLoRA weights specifically
trainer.model.save_pretrained("Qwen-1.5B-QLoRA")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,3.868610
2,2.830005
3,2.695863
4,3.106357
5,2.231497
6,2.246057
7,0.720175
8,1.629687
9,1.469296
10,1.133720


In [ ]:
from peft import PeftModel
import torch
import gc

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# ---------------------------------------------------------
# 1. Merge standard LoRA Model
# ---------------------------------------------------------
print("Merging LoRA model...")

# Load base model in 16-bit precision[cite: 1]
base_model_lora = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Load the LoRA adapter[cite: 1]
lora_model = PeftModel.from_pretrained(base_model_lora, "Qwen-1.5B-LoRA")

# Merge and unload
merged_lora = lora_model.merge_and_unload()

# Save the merged model and tokenizer
merged_lora.save_pretrained("Qwen-1.5B-LoRA-Merged")
tokenizer.save_pretrained("Qwen-1.5B-LoRA-Merged")

print("LoRA merge complete and saved to 'Qwen-1.5B-LoRA-Merged'.")

# ---------------------------------------------------------
# Clean up memory to avoid OOM (Out Of Memory) errors
# ---------------------------------------------------------
del base_model_lora
del lora_model
del merged_lora
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------
# 2. Merge QLoRA Model
# ---------------------------------------------------------
print("Merging QLoRA model...")

# CRITICAL: Load base model in 16-bit precision again, NOT in 4-bit[cite: 1]
base_model_qlora = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Load the QLoRA adapter[cite: 1]
qlora_model = PeftModel.from_pretrained(base_model_qlora, "Qwen-1.5B-QLoRA")

# Merge and unload
merged_qlora = qlora_model.merge_and_unload()

# Save the merged model and tokenizer
merged_qlora.save_pretrained("Qwen-1.5B-QLoRA-Merged")
tokenizer.save_pretrained("Qwen-1.5B-QLoRA-Merged")

print("QLoRA merge complete and saved to 'Qwen-1.5B-QLoRA-Merged'.")

Merging LoRA model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

LoRA merge complete and saved to 'Qwen-1.5B-LoRA-Merged'.
Merging QLoRA model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

QLoRA merge complete and saved to 'Qwen-1.5B-QLoRA-Merged'.


In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

def generate_response(model, tokenizer, prompt_text):
    """Formats the input into Qwen's chat template and generates a response."""
    messages = [
        {"role": "system", "content": "You are a personalized AI assistant representing Abdelrahman ElKhatteeb."},
        {"role": "user", "content": prompt_text}
    ]

    # Apply Qwen's native chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only the generated response tokens
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


def clear_gpu():
    """Flushes GPU VRAM between model loads."""
    gc.collect()
    torch.cuda.empty_cache()


# Define your test question
test_prompt = "Who are you and what is your background?"

# ==========================================
# 1. Test Standard LoRA Adapter
# ==========================================
print("--- Loading Standard LoRA Model ---")
base_model_lora = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
lora_model = PeftModel.from_pretrained(base_model_lora, "Qwen-1.5B-LoRA")
lora_model.eval()

print(f"\nUser: {test_prompt}")
lora_output = generate_response(lora_model, tokenizer, test_prompt)
print(f"LoRA Output:\n{lora_output}\n")

# Cleanup memory before loading the next model
del base_model_lora
del lora_model
clear_gpu()

# ==========================================
# 2. Test QLoRA Adapter (Loaded in 4-bit)
# ==========================================
print("--- Loading QLoRA Model ---")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model_qlora = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
qlora_model = PeftModel.from_pretrained(base_model_qlora, "Qwen-1.5B-QLoRA")
qlora_model.eval()

print(f"\nUser: {test_prompt}")
qlora_output = generate_response(qlora_model, tokenizer, test_prompt)
print(f"QLoRA Output:\n{qlora_output}\n")

--- Loading Standard LoRA Model ---


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


User: Who are you and what is your background?
LoRA Output:
I am Abdelrahman ElKhatteeb. I'm an Iraqi software developer based in Baghdad, with experience from 2019 to the current day.

--- Loading QLoRA Model ---


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


User: Who are you and what is your background?
QLoRA Output:
I am Abdelrahman ElKhatteeb. I am a student at Cairo University pursuing a Bachelor's degree in Computer Science, with an expected graduation in June 2027.



In [ ]:
# Copy the LoRA merged model
!cp -r /content/Qwen-1.5B-LoRA-Merged /content/drive/MyDrive/

# Copy the QLoRA merged model
!cp -r /content/Qwen-1.5B-QLoRA-Merged /content/drive/MyDrive/

In [ ]:
from huggingface_hub import login

# It will prompt you for your Hugging Face Write Token
login()



In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_name = "abdelrhmanakh/Qwen-1.5B-QLoRA-Merged"

print(f"Pushing model to {repo_name}...")
api.create_repo(repo_name, exist_ok=True)

# Upload the entire folder
api.upload_folder(
    folder_path="./Qwen-1.5B-QLoRA-Merged", # The path to your saved folder
    repo_id=repo_name,
    repo_type="model"
)

Pushing model to abdelrhmanakh/Qwen-1.5B-QLoRA-Merged...


CommitInfo(commit_url='https://huggingface.co/abdelrhmanakh/Qwen-1.5B-QLoRA-Merged/commit/6c0b067a05b814440cbe02104457b5db76d8022e', commit_message='Upload folder using huggingface_hub', commit_description='', oid='6c0b067a05b814440cbe02104457b5db76d8022e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/abdelrhmanakh/Qwen-1.5B-QLoRA-Merged', endpoint='https://huggingface.co', repo_type='model', repo_id='abdelrhmanakh/Qwen-1.5B-QLoRA-Merged'), pr_revision=None, pr_num=None)